# Import Variables and Load Packages

In [ ]:
import uproot
#print("uproot version: ", uproot.__version__)
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.notebook import tqdm
import pickle
from collections import Counter
from particle import Particle
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import awkward as ak

f = uproot.open('/Users/katherinepulido/Desktop/H-KPMTcalibration/HKPMTcalibration/LIGen395_Pos0_DROff_hits_flat-2.root')

print(f['photonTree'].keys())
f['hitsTree'].keys()


## Load Variables

In [ ]:
photon = ['event', 
          'startX', 
          'startY', 
          'startZ', 
          'endX', 
          'endY', 
          'endZ'
        ]

hits = ['event', 'subevent', 'pmt_id', 'charge', 'time', 'posX', 'posY', 'posZ']

vars = {}
vars2 = {}
vars.update(f["photonTree"].arrays(photon, library="np"))
vars2.update(f["hitsTree"].arrays(hits, library="np"))

ph_df = pd.DataFrame(vars)
hit_df = pd.DataFrame(vars2)

In [ ]:
# some global variables

radius = 3242.76611328125
height = 2*3296.47119140625

xmin = -3242.76611328125
xmax = 3242.76611328125
ymin = -3242.76611328125
ymax = 3242.76611328125
zmin = -3296.47119140625
zmax = 3296.47119140625

# can one photon hit on two pmts?

In [ ]:
# make a pmt dicitonary

pmt_dict = {
    row.pmt_id: (row.posX, row.posY, row.posZ)
    for row in hit_df[['pmt_id', 'posX', 'posY', 'posZ']]
        .drop_duplicates(subset='pmt_id')
        .itertuples(index=False)
}

pmt_dict[1234]

pmt_normals = {}
for row in hit_df[['pmt_id', 'posX', 'posY', 'posZ']].drop_duplicates(subset='pmt_id').itertuples(index=False):
    pmt_id, pmt_x, pmt_y, pmt_z = row.pmt_id, row.posX, row.posY, row.posZ

    # define normal vector
    if pmt_z == 3296.47119140625:
        normal_vector = np.array([0, 0, -1])
    elif pmt_z == -3296.47119140625:
        normal_vector = np.array([0, 0, 1])
    else:
        normal_vector = np.array([-pmt_x/radius, -pmt_y/radius, 0])

    # normalize
    normal_vector = normal_vector / np.linalg.norm(normal_vector)

    # store in separate dict
    pmt_normals[pmt_id] = normal_vector
    


In [ ]:
def make_display(event_idx):    
    ax = plt.figure(figsize=(12,12)).add_subplot(projection='3d')

    #cylinder
    radius = 3242.76611328125
    z = np.linspace(-3296.47119140625, 3296.47119140625, 1000)
    theta = np.linspace(0, 2*np.pi, 1000)
    Theta, Zc = np.meshgrid(theta, z)
    Xc = radius * np.cos(Theta)
    Yc = radius * -np.sin(Theta)
    # cyl parameters
    rstride = 20
    cstride = 10
    ax.plot_surface(Xc, Yc, Zc, alpha=0.1, rstride=rstride, cstride=cstride, color='grey')
    ax.plot_surface(Xc, -Yc, Zc, alpha=0.1, rstride=rstride, cstride=cstride, color='grey')

    # add source positions
    source_positions = {
        0: (3220, 0, -2465.66),
        4: (3220, 0, -1643.78),
        8: (3220, 0, -821.88),
        12: (3220, 0, 0),
        16: (3220, 0, 821.888),
        20: (3220, 0, 1643.78),
        24: (3220, 0, 2465.66),
    }

    for key, (x, y, z) in source_positions.items():
        ax.scatter(x, y, z, label=f"Source Idx: {key}", s=100)

    event_ph_df = ph_df[ph_df["event"] == event_idx]
    event_hit_df = hit_df[hit_df["event"] == event_idx]
    

    # Apply filter
    y_df = event_ph_df[event_ph_df["startX"] != event_ph_df["endX"]]
    #y_df = ph_df.sample(n=10000)

    # Only take 2000 rows
    #y_df = y_df.iloc[:50000]

    print(y_df.shape[0])


    # Start points
    #x_starts = np.full(y_df.shape[0], 3220)
    #y_starts = np.zeros(y_df.shape[0])
    #z_starts = np.full(y_df.shape[0], -2465.66)

    x_starts = y_df["startX"].values * 0.1
    y_starts = y_df["startY"].values * 0.1
    z_starts = y_df["startZ"].values * 0.1

    # End points
    x_ends = y_df["endX"].values * 0.1
    y_ends = y_df["endY"].values * 0.1
    z_ends = y_df["endZ"].values * 0.1
        
    pmt_xs = event_hit_df['posX']
    pmt_ys = event_hit_df['posY']
    pmt_zs = event_hit_df['posZ']
    pmt_charges = event_hit_df['charge']
    #event_hit_df['posX']

    # Plot line segments
    for xs, ys, zs, xe, ye, ze in zip(x_starts, y_starts, z_starts, x_ends, y_ends, z_ends):
        ax.plot([xs, xe], [ys, ye], [zs, ze], color='blue', alpha=0.2)
        for pmtx, pmty, pmtz, pmte, in zip(pmt_xs, pmt_ys, pmt_zs, pmt_charges):
            if np.linalg.norm(np.array([xe, ye, ze]) - np.array([pmtx, pmty, pmtz])) < 26.95:
                ax.scatter(pmtx, pmty, pmtz, color='red')
                
    pmt_hit_df = hit_df[hit_df["pmt_id"]==19343]

    #x1 = pmt_hit_df["posX"].iloc[0] 
    #y1 = pmt_hit_df["posY"].iloc[0] 
    #z1 = pmt_hit_df["posZ"].iloc[0] 
 
    pmt_410 = hit_df[hit_df["pmt_id"] == 410]
    
    x1 = pmt_410["posX"]
    y1 = pmt_410["posY"]
    z1 = pmt_410["posZ"]
    
    
    
    ax.scatter(x1, y1, z1, color='purple', label='pmt 410', marker="o", s=250)

    ax.set_xlabel("X (cm)")
    ax.set_ylabel("Y (cm)")
    ax.set_zlabel("Z (cm)")

    ax.legend()
    plt.title(f'Photon Tracks with PMT Hits for Event {event_idx}')

    plt.show()

make_display(1)

# make version for one PMT with angular dependence
# look for PMT with highest amt of hits
# hit count ratio as a function of incident angle



In [ ]:
#vars.update(f["photonTree"].arrays(photon, library="np"))
#ph_df = pd.DataFrame(vars)

print(ph_df.shape[0])

moved_df = ph_df[ph_df["startX"] != ph_df["endX"]]

moved_df.shape[0]

hit_df.shape[0]





In [ ]:
top10 = hit_df["pmt_id"].value_counts().head(10)

for pmt_id, freq in top10.items():
    print(f"pmt_id {pmt_id}: {freq}")
    
top10_id = ph_df["closest_pmt"].value_counts().head(10)

for pmt_id, freq in top10_id.items():
    print(f"pmt_id {pmt_id}: {freq}")
    


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def make_display_single_pmt(pmt_id):    
    
    # Filter hits for this PMT
    one_pmt_df = hit_df[hit_df["pmt_id"] == pmt_id]
    
    # If photons are already matched, otherwise you probably want one_pmt_df
    #y_df = ph_df[ph_df["closest_pmt"]==pmt_id]
                
    y_df = assign_photons_to_single_pmt(ph_df, pmt_dict, pmt_id, max_dist=3000.0)     
           
    print(y_df.shape[0])

    # End points (scaled to cm)
    x_ends = y_df["endX"].values * 0.1
    y_ends = y_df["endY"].values * 0.1
    z_ends = y_df["endZ"].values * 0.1
        
    # PMT position
    pmt_x, pmt_y, pmt_z = pmt_dict[pmt_id]
    
    # Classify PMT type
    if np.isclose(pmt_z, 3296.47119140625, atol=1):
        pmt_type = 'barrel'
    else:
        pmt_type = 'endcap'
    
    fig, ax = plt.subplots(figsize=(6,6))

    # PMT face
    circle = patches.Circle((0, 0), radius=25.4, color='r', fill=True, alpha=0.2)
    ax.add_patch(circle)

    # --- Barrel reference angle ---
    if pmt_type == 'barrel':
        phi0 = np.arctan2(pmt_y, pmt_x)
        R = np.sqrt(pmt_x**2 + pmt_y**2)

    # --- Loop over hits ---
    for xe, ye, ze in zip(x_ends, y_ends, z_ends):

        # Only keep hits close to PMT face
        print(np.linalg.norm([xe-pmt_x, ye-pmt_y, ze-pmt_z]))
        if np.linalg.norm([xe - pmt_x, ye - pmt_y, ze - pmt_z]) < 300:

            dx = xe - pmt_x
            dy = ye - pmt_y
            dz = ze - pmt_z

            if pmt_type == 'barrel':
                # Cylindrical projection

                phi = np.arctan2(ye, xe)
                dphi = phi - phi0

                # Wrap angle to [-pi, pi]
                dphi = (dphi + np.pi) % (2*np.pi) - np.pi

                x_proj = R * dphi      # arc length (r * theta)
                y_proj = dz            # vertical

            else:
                # Endcap: flat projection
                x_proj = dx
                y_proj = dy

            ax.scatter(x_proj, y_proj, s=5, alpha=0.6)
            print(f"plotted at {x_proj, y_proj} ")

    # Formatting
    ax.set_aspect('equal')
    ax.set_xlim(-30, 30)
    ax.set_ylim(-30, 30)

    if pmt_type == 'barrel':
        ax.set_xlabel("r·Δθ (cm)")
        ax.set_ylabel("Δz (cm)")
    else:
        ax.set_xlabel("Δx (cm)")
        ax.set_ylabel("Δy (cm)")

    ax.set_title(f"PMT {pmt_id} ({pmt_type}) Hit Map")

    plt.show()


# Example
make_display_single_pmt(21)

In [ ]:
ph_df[ph_df["closest_pmt"]==14781]

In [ ]:
ph_df.to_csv("ph_df_with_closest_pmt.csv", index=False)

In [ ]:

def make_display(event_idx):    
    ax = plt.figure(figsize=(12,12)).add_subplot(projection='3d')

    #cylinder
    radius = 3242.76611328125
    z = np.linspace(-3296.47119140625, 3296.47119140625, 1000)
    theta = np.linspace(0, 2*np.pi, 1000)
    Theta, Zc = np.meshgrid(theta, z)
    Xc = radius * np.cos(Theta)
    Yc = radius * -np.sin(Theta)
    # cyl parameters
    rstride = 20
    cstride = 10
    ax.plot_surface(Xc, Yc, Zc, alpha=0.1, rstride=rstride, cstride=cstride, color='grey')
    ax.plot_surface(Xc, -Yc, Zc, alpha=0.1, rstride=rstride, cstride=cstride, color='grey')

    # add source positions
    source_positions = {
        0: (3220, 0, -2465.66),
        4: (3220, 0, -1643.78),
        8: (3220, 0, -821.88),
        12: (3220, 0, 0),
        16: (3220, 0, 821.888),
        20: (3220, 0, 1643.78),
        24: (3220, 0, 2465.66),
    }

    for key, (x, y, z) in source_positions.items():
        ax.scatter(x, y, z, label=f"Source Idx: {key}", s=100)

    event_ph_df = ph_df[ph_df["event"] == event_idx]
    event_hit_df = hit_df[hit_df["event"] == event_idx]

    # Apply filter
    y_df = event_ph_df
    #y_df = ph_df.sample(n=10000)

    # Only take 2000 rows
    #y_df = y_df.iloc[:50000]

    print(y_df.shape[0])


    # Start points
    #x_starts = np.full(y_df.shape[0], 3220)
    #y_starts = np.zeros(y_df.shape[0])
    #z_starts = np.full(y_df.shape[0], -2465.66)

    x_starts = y_df["startX"].values * 0.1
    y_starts = y_df["startY"].values * 0.1
    z_starts = y_df["startZ"].values * 0.1

    # End points
    x_ends = y_df["endX"].values * 0.1
    y_ends = y_df["endY"].values * 0.1
    z_ends = y_df["endZ"].values * 0.1
        
    pmt_xs = event_hit_df['posX']
    pmt_ys = event_hit_df['posY']
    pmt_zs = event_hit_df['posZ']
    pmt_charges = event_hit_df['charge']
    #event_hit_df['posX']

    # Plot line segments
    for xs, ys, zs, xe, ye, ze in zip(x_starts, y_starts, z_starts, x_ends, y_ends, z_ends):
        ax.plot([xs, xe], [ys, ye], [zs, ze], color='blue', alpha=0.2)
        for pmtx, pmty, pmtz, pmte, in zip(pmt_xs, pmt_ys, pmt_zs, pmt_charges):
            if np.linalg.norm(np.array([xe, ye, ze]) - np.array([pmtx, pmty, pmtz])) < 26.95:
                ax.scatter(pmtx, pmty, pmtz, color='red')
                
    pmt_hit_df = hit_df[hit_df["pmt_id"]==19343]

    #x1 = pmt_hit_df["posX"].iloc[0] 
    #y1 = pmt_hit_df["posY"].iloc[0] 
    #z1 = pmt_hit_df["posZ"].iloc[0] 
 
    pmt_410 = hit_df[hit_df["pmt_id"] == 410]
    
    x1 = pmt_410["posX"]
    y1 = pmt_410["posY"]
    z1 = pmt_410["posZ"]
    
    
    
    ax.scatter(x1, y1, z1, color='purple', label='pmt 410', marker="o", s=250)

    ax.set_xlabel("X (cm)")
    ax.set_ylabel("Y (cm)")
    ax.set_zlabel("Z (cm)")

    ax.legend()
    plt.title(f'Photon Tracks with PMT Hits for Event {event_idx}')

    plt.show()

make_display(1)

# make version for one PMT with angular dependence
# look for PMT with highest amt of hits
# hit count ratio as a function of incident angle



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_single_pmt_hits(pmt_id):
    """
    3D plot of all photon endpoints assigned to a single PMT.
    """
    ax = plt.figure(figsize=(10, 10)).add_subplot(111, projection='3d')
    
    #cylinder
    radius = 3242.76611328125
    z = np.linspace(-3296.47119140625, 3296.47119140625, 1000)
    theta = np.linspace(0, 2*np.pi, 1000)
    Theta, Zc = np.meshgrid(theta, z)
    Xc = radius * np.cos(Theta)
    Yc = radius * -np.sin(Theta)
    # cyl parameters
    rstride = 20
    cstride = 10
    ax.plot_surface(Xc, Yc, Zc, alpha=0.1, rstride=rstride, cstride=cstride, color='grey')
    ax.plot_surface(Xc, -Yc, Zc, alpha=0.1, rstride=rstride, cstride=cstride, color='grey')

    # add source positions
    source_positions = {
        0: (3220, 0, -2465.66),
        4: (3220, 0, -1643.78),
        8: (3220, 0, -821.88),
        12: (3220, 0, 0),
        16: (3220, 0, 821.888),
        20: (3220, 0, 1643.78),
        24: (3220, 0, 2465.66),
    }

    for key, (x, y, z) in source_positions.items():
        ax.scatter(x, y, z, label=f"Source Idx: {key}", s=100)
    # Filter photons closest to this PMT
    y_df = assign_photons_to_single_pmt(ph_df, pmt_dict, pmt_id, max_dist=30.0)
    print(f"Number of photons for PMT {pmt_id}: {y_df.shape[0]}")

    pmt_x, pmt_y, pmt_z = pmt_dict[pmt_id][0],pmt_dict[pmt_id][1],pmt_dict[pmt_id][2]

    # Photon endpoints (scaled to cm)
    x_ends = y_df["endX"].values * 0.1
    y_ends = y_df["endY"].values * 0.1
    z_ends = y_df["endZ"].values * 0.1

    # Plot PMT position
    ax.scatter(pmt_x, pmt_y, pmt_z, color='red', s=200, label=f"PMT {pmt_id}")

    # Plot all photon endpoints assigned to this PMT
    ax.scatter(x_ends, y_ends, z_ends, color='blue', alpha=0.2, s=5, label="Photon endpoints")

    # Optional: draw lines from PMT to each photon (comment out if too many points)
    # for xe, ye, ze in zip(x_ends, y_ends, z_ends):
    #     ax.plot([pmt_x, xe], [pmt_y, ye], [pmt_z, ze], color='lightblue', alpha=0.05)

    ax.set_xlabel("X (cm)")
    ax.set_ylabel("Y (cm)")
    ax.set_zlabel("Z (cm)")
    ax.set_title(f"Photons assigned to PMT {pmt_id}")
    ax.legend()
    plt.show()
    


plot_single_pmt_hits(9000)

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import time

def assign_pmts_by_distance(ph_df, pmt_dict, max_dist=30.0):
    """
    Assign PMTs to photons only if the photon end is within `max_dist` cm.
    Shows a progress bar while looping over PMTs.
    """
    start_time = time.time()
    
    # Photon endpoints (in cm; multiply if your df is in mm)
    ph_points = ph_df[['endX','endY','endZ']].values
    
    # PMT positions
    pmt_ids = np.array(list(pmt_dict.keys()))
    
    # Initialize result column with -1 (no PMT assigned)
    closest_pmts = np.full(len(ph_df), -1, dtype=int)
    
    # Loop over PMTs with tqdm progress bar
    for pmt_id, (px, py, pz) in tqdm(pmt_dict.items(), desc="Assigning PMTs", total=len(pmt_dict)):
        dx = ph_points[:,0] - px
        dy = ph_points[:,1] - py
        dz = ph_points[:,2] - pz
        dist = np.sqrt(dx**2 + dy**2 + dz**2)
        
        # Assign PMT if within max_dist and not already assigned
        mask = (dist <= max_dist) & (closest_pmts == -1)
        closest_pmts[mask] = pmt_id

    ph_df['closest_pmt'] = closest_pmts
    
    elapsed = time.time() - start_time
    print(f"Finished assigning PMTs in {elapsed:.2f} seconds")
    
    return ph_df

ph_attempt_2 = assign_pmts_by_distance(ph_df, pmt_dict)


In [ ]:
import numpy as np
import pandas as pd

def assign_photons_to_single_pmt(ph_df, pmt_dict, pmt_id, max_dist=30.0):
    """
    Assign photons to a single PMT if within max_dist cm.
    
    Parameters
    ----------
    ph_df : pd.DataFrame
        Must contain 'endX','endY','endZ' (cm or mm).
    pmt_dict : dict
        {pmt_id: (x, y, z)} in same units as ph_df.
    pmt_id : int
        PMT to assign photons to.
    max_dist : float
        Maximum distance for assignment (cm).
    
    Returns
    -------
    filtered_ph_df : pd.DataFrame
        Subset of photons assigned to this PMT.
    """
    if pmt_id not in pmt_dict:
        raise ValueError(f"PMT {pmt_id} not found in pmt_dict.")
    
    px, py, pz = pmt_dict[pmt_id]
    
    # Photon endpoints
    x = ph_df['endX'].values * .1
    y = ph_df['endY'].values *.1 
    z = ph_df['endZ'].values *.1
    
    # Compute distance to this PMT
    dist = np.sqrt((x - px)**2 + (y - py)**2 + (z - pz)**2)
    
    # Filter photons within max_dist
    mask = dist <= max_dist
    filtered_ph_df = ph_df[mask].copy()
    
    # Optionally, assign PMT ID to column
    filtered_ph_df['closest_pmt'] = pmt_id
    
    print(f"Assigned {mask.sum()} photons to PMT {pmt_id}")
    
    return filtered_ph_df